# LangGraph 01 · 状态图（StateGraph）

这是整个 `01_langgraph/` 的第一课。目标只有一个：**把「智能体」看成一张图**，
并亲手拼出第一张能跑的图。

全节的四个概念，后面每一课都会反复出现：

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 状态 State | 在图里流转的共享数据 | `class MyState(TypedDict)` |
| 节点 Node | 一个函数：收 state → 返回增量 | `step_one` / `step_two` |
| 普通边 Edge | 固定从 A 跳到 B | `builder.add_edge("step_one", "step_two")` |
| 条件边 Conditional Edge | 运行时按返回值决定去哪 | `builder.add_conditional_edges(...)` |

> **本 notebook 由 `Agent/01_langgraph/` 下 4 个脚本合并而成**：
> `01_基础图.py`（课案原版 80 行）、`01_基础图_jxsd.py`（完整版 275 行），
> 以及后续扩建要合并的 `00_框架总览_jxsd.py`、`10_控制流与函数式API_官方补充.py`。

**官方文档**
- 图 API 参考：<https://docs.langchain.com/oss/python/langgraph/graph-api>
- 快速上手：<https://docs.langchain.com/oss/python/langgraph/quickstart>
- 用图 API 构建：<https://docs.langchain.com/oss/python/langgraph/use-graph-api>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟢 运行档位 | **离线可跑** —— 不读 `.env`、不连大模型、不起任何服务 |
| 依赖 | `langgraph`（本项目 venv 已装） |
| 预计耗时 | < 2 秒 |
| 前置服务 | 无 |

> 这是 `01_langgraph/` 里**唯一能离线秒跑**的一课 —— 因为它只演示图本身的调度，
> 还没有任何「记忆 / 模型 / 工具」参与。后面的课会逐步把这三样加进来。

## 本节地图

先看这一课在整章里的位置，以及一张图内部的数据怎么流。

```mermaid
graph LR
    A["START<br/>图入口"] --> B["step_one<br/>count + 1"]
    B --> C["step_two<br/>count × 2"]
    C --> D{"choose_path<br/>count > 10 ?"}
    D -->|"是"| E["big<br/>太大了"]
    D -->|"否"| F["small<br/>很小"]
    E --> G["END<br/>图出口"]
    F --> G
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 从 | 到 | 边的类型 | 什么时候走 |
|---|---|---|---|
| `START` | `step_one` | 普通边 | 总是 |
| `step_one` | `step_two` | 普通边 | 总是 |
| `step_two` | `big` 或 `small` | **条件边** | 由 `choose_path` 的返回值决定 |
| `big` / `small` | `END` | 普通边 | 总是 |

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课其实用不到 `config`，但这一格仍然保留 —— 一是保持全仓统一，
> 二是它顺便给出了 `NB_DIR` / `WORKDIR` 两个变量，后续课时要用它来定位临时文件。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 1. 课案原版：最小的一张图

课案原版只有 80 行，正好够把四个概念各出现一次。
**先看最短的实现，再看完整版**，两者的差距就是本课要讲的全部内容。

In [ ]:
# ---------- 1.1 状态：一个普通的 TypedDict ----------
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    """
    状态就是一个 TypedDict，所有节点共享。

    - messages：普通覆盖式字段（后写的值直接覆盖先写的）
    - history ：使用 Annotated + operator.add 实现「追加式」字段
                每个节点返回的列表会被拼接（reduce）到原有列表后面
    """
    messages: str
    history: Annotated[list[str], operator.add]

In [ ]:
# ---------- 1.2 节点：就是普通函数 ----------
def node_a(state: State) -> dict:
    """节点 A：接收当前状态，返回对状态的「增量更新」（而不是全量状态）"""
    print(f"[节点A] 收到消息：{state['messages']}")
    # 只需要返回要更新的字段
    return {"messages": "A 已处理", "history": ["经过节点A"]}


def node_b(state: State) -> dict:
    """节点 B：可以读到前面节点写入的数据"""
    print(f"[节点B] 收到消息：{state['messages']}")
    return {"messages": "B 已处理", "history": ["经过节点B"]}

In [ ]:
# ---------- 1.3 条件边：根据状态动态决定下一个节点 ----------
def route(state: State) -> str:
    """返回值是「下一个节点」的名字，也可以返回 END 表示结束"""
    if "B" in state["messages"]:
        return END
    return "b"

In [ ]:
# ---------- 1.4 组装图 ----------
builder = StateGraph(State)

# 注册节点
builder.add_node("a", node_a)
builder.add_node("b", node_b)

# 固定边：START -> a
builder.add_edge(START, "a")
# 条件边：a 之后走 route 函数决定去向
builder.add_conditional_edges("a", route, ["b", END])
# 固定边：b -> END
builder.add_edge("b", END)

# 编译：把 builder 变成可执行的图
graph = builder.compile()

In [ ]:
# ---------- 1.5 执行 ----------
# invoke：一次性执行完整张图，返回最终状态
result = graph.invoke({"messages": "你好"})
print("最终状态：", result)

### 预期输出

```text
[节点A] 收到消息：你好
[节点B] 收到消息：A 已处理
最终状态： {'messages': 'B 已处理', 'history': ['经过节点A', '经过节点B']}
```

**三个值得停一下看的细节：**

1. 节点 B 收到的是 `"A 已处理"` 而不是 `"你好"` —— 说明**节点 A 的返回值真的并进了状态**；
2. `history` 里两条都在 —— 因为字段声明用了 `Annotated[..., operator.add]`（追加式），
   而 `messages` 只剩最后一条（覆盖式）；
3. 条件边走的是 `"b"` 而不是 `END` —— 因为此刻 `state["messages"]` 是 `"A 已处理"`，
   里面不含字母 `"B"`。

## 2. 完整版：每个元素都正式定义一遍

课案原版把状态、节点、边混在一个文件里；完整版把它们**按职责切开**，
这样后面加节点、加分支时不会互相干扰。

### 2.1 状态：图的「共享内存」

`TypedDict` 只描述「有哪些字段、什么类型」，**不产生运行时校验**。
它的真正作用有两个：

- 给编辑器做补全与类型检查；
- 告诉 LangGraph **每个字段该怎么合并**（这才是关键，见 2.5 与第 3 节）。

In [ ]:
# ============================================================
# 1. 定义状态（图中流转的数据）
# ============================================================
class MyState(TypedDict):
    """课案原文的状态定义。

    TypedDict 只描述「有哪些字段、什么类型」，不产生运行时校验，
    它的作用是给编辑器 / LangGraph 提供字段清单和合并规则。
    """

    count: int  # 计数器：普通字段，节点返回什么就是什么（覆盖式）
    log: str  # 日志：普通字段，每个节点覆盖写入

### 2.2 节点：返回「增量」，不是「全量」

这是新手最容易搞错的一点：节点函数**只需要返回自己改动的字段**。
返回完整 state 不会报错，但会让 reducer 的语义变得难以预料。

In [ ]:
# ============================================================
# 2. 定义节点函数（接收 state，返回更新的 state）
# ============================================================
def step_one(state: MyState) -> dict:
    """步骤一：count + 1，并把日志重置为「执行了步骤一」。"""
    # 注意只返回要改的字段。log 这里是覆盖，不是拼接——
    # 想拼接（保留历史）就得用 Annotated + operator.add，见第 5 节。
    return {"count": state["count"] + 1, "log": "执行了步骤一"}


def step_two(state: MyState) -> dict:
    """步骤二：count × 2，并把新日志拼在旧日志后面（手工拼字符串）。"""
    return {"count": state["count"] * 2, "log": state["log"] + " → 步骤二"}

### 2.3 路由函数：只返回「下一个节点的名字」

⚠️ **路由函数不执行跳转，也不改状态**。真正查表跳转的是 LangGraph 自己。
这条边界想清楚了，条件边就不会写错。

In [ ]:
def choose_path(state: MyState) -> str:
    """条件边的路由函数。

    ⚠️ 路由函数只做一件事：**返回「下一个节点的名字」**（字符串）。
    它不执行跳转，也不碰状态——真正的跳转由 LangGraph 按
    add_conditional_edges 里给的映射表完成。
    """
    # 条件边：根据 count 值决定走哪条路
    return "big" if state["count"] > 10 else "small"


def big_handler(state: MyState) -> dict:
    """大数分支：count > 10 时走这里。"""
    return {"log": f"count={state['count']}, 太大了"}


def small_handler(state: MyState) -> dict:
    """小数分支：count <= 10 时走这里。"""
    return {"log": f"count={state['count']}, 很小"}

### 2.4 组装图

`add_edge(START, "step_one")` 里的 `START` / `END` 是 LangGraph 的**虚拟节点**：
它们不对应任何函数，只标记「从哪进、到哪出」。

In [ ]:
# ============================================================
# 3. 构建图（课案原文的组装流程）
# ============================================================
builder = StateGraph(MyState)

builder.add_node("step_one", step_one)  # 添加节点：节点名 "step_one" → 函数 step_one
builder.add_node("step_two", step_two)
builder.add_node("big", big_handler)
builder.add_node("small", small_handler)

builder.add_edge(START, "step_one")  # 普通边：START → step_one（START 是图的虚拟入口）
builder.add_edge("step_one", "step_two")  # 普通边：step_one → step_two
builder.add_edge("big", END)  # big → 结束（END 是图的虚拟出口）
builder.add_edge("small", END)  # small → 结束

### 2.5 条件边：本节的绝对重点

`add_conditional_edges` 的第三个参数有**三种写法**，行为并不一样：

| 写法 | 例子 | 语义 |
|---|---|---|
| ① 列表 | `["big", "small"]` | 路由函数返回 `"big"`，就必须存在**同名节点** `big` |
| ② 映射字典（推荐） | `{"big": "big", "small": "small"}` | 左边是返回值，右边是真正跳的节点名；**两者可以不同名** |
| ③ 不传第三个参数 | — | LangGraph 自行推断目标，少写代码但图结构不直观 |

为什么推荐映射字典？因为它把「路由函数的返回值」和「节点名」解耦了，
例如 `{"big": "handle_large"}` 完全合法 —— 返回值只是**内部的暗号**。

> 第三个参数还有一个隐藏作用：**声明这笔分叉可能去哪些地方**。
> 于是 `graph.get_graph()` 才能把条件边画出来 —— 不传，图就是残缺的。
> 这一点在下面的「打印图结构」里会亲眼看到。

In [ ]:
# ---------- 3.1 条件边：本节的绝对重点 ----------
# add_conditional_edges 的三个参数，课案里给了「列表」写法，
# 这里特意用「映射字典」写法，把两者的区别讲透：
#
#   builder.add_conditional_edges("step_two", choose_path, {"big": "big", "small": "small"})
#                                 ↑ 源节点    ↑ 路由函数   ↑ 路由函数返回值 → 真实节点名
builder.add_conditional_edges(
    "step_two",
    choose_path,  # 路由函数，返回目标节点名
    {"big": "big", "small": "small"},  # 返回值 → 节点映射
)

graph = builder.compile()  # 编译：把 builder 变成可执行的图（编译后不能再改结构）
print("图已编译：", graph)

### 2.6 跑起来：两条分支都要走一遍

只跑一条分支是看不出条件边生效的 —— 换个初值，让 `count` 越过 10，走另一条路。

In [ ]:
# ============================================================
# 4. 运行（课案原文的 invoke）
# ============================================================
def run_course_example() -> None:
    """课案原文示例：count=1 → +1=2 → ×2=4 → small → "count=4, 很小"。"""
    print("=" * 72)
    print("① 课案原文示例：invoke({'count': 1, 'log': ''})")
    print("=" * 72)
    print("  数据流：count=1 —step_one→ 2 —step_two→ 4 —choose_path→ small")
    result = graph.invoke({"count": 1, "log": ""})
    print("  返回：", result)
    # 预期输出：{'count': 4, 'log': 'count=4, 很小'}
    print()


run_course_example()

### 预期输出

```text
① 课案原文示例：invoke({'count': 1, 'log': ''})
  数据流：count=1 —step_one→ 2 —step_two→ 4 —choose_path→ small
  返回： {'count': 4, 'log': 'count=4, 很小'}
```

课案注释里写的那条数据流（`count=1 → +1=2 → ×2=4 → small`）**本机实测完全一致**。

In [ ]:
def run_big_branch() -> None:
    """换个初值走 big 分支：count=6 → +1=7 → ×2=14 > 10 → big。"""
    print("=" * 72)
    print("② 换一个初值，让条件边走另一条分叉：invoke({'count': 6, 'log': ''})")
    print("=" * 72)
    print("  数据流：count=6 —step_one→ 7 —step_two→ 14 > 10 —choose_path→ big")
    result = graph.invoke({"count": 6, "log": ""})
    print("  返回：", result)
    # 预期输出：{'count': 14, 'log': 'count=14, 太大了'}
    print()


run_big_branch()

### 预期输出

```text
② 换一个初值，让条件边走另一条分叉：invoke({'count': 6, 'log': ''})
  数据流：count=6 —step_one→ 7 —step_two→ 14 > 10 —choose_path→ big
  返回： {'count': 14, 'log': 'count=14, 太大了'}
```

**同一个图、同一段代码，只换了初值，走的就是另一条边** —— 这就是条件边的意义。

In [ ]:
def show_graph_structure() -> None:
    """把图的结构打印出来——让「节点/普通边/条件边」看得见摸得着。"""
    print("=" * 72)
    print("③ 图的结构（get_graph()）：普通边与条件边的区别一目了然")
    print("=" * 72)
    net = graph.get_graph()
    for edge in sorted(net.edges, key=lambda e: (e.source, e.target)):
        kind = "条件边" if edge.conditional else "普通边"
        print(f"  [{kind}] {edge.source:<10} → {edge.target}")
    print()


show_graph_structure()

### 预期输出

```text
  [普通边] __start__  → step_one
  [普通边] big        → __end__
  [普通边] small      → __end__
  [普通边] step_one   → step_two
  [条件边] step_two   → big
  [条件边] step_two   → small
```

这里有两个「原来如此」：

- `START` / `END` 在打印里叫 `__start__` / `__end__` —— 它们是 LangGraph
  内部插入的虚拟节点，所以**永远出现在边表里**，但你没有写进 `add_node`；
- `step_two` 出现了**两条**出边，且都标着「条件边」—— 正是第三个参数
  （那个映射字典）声明出来的。**把第三个参数去掉，这两行就会消失**，
  图变成残缺的。

## 3. 追加式字段：`Annotated` + `operator.add`

上面 `MyState.log` 是**覆盖式**的。之所以还能看到历史，是因为我们在
`step_two` 里**手工**把字符串拼了起来（`state["log"] + " → 步骤二"`）。

节点一多，每个都要手工拼接，又啰嗦又容易漏。LangGraph 的解法是
「声明式 reducer」：给字段挂一个**合并函数**，让框架自动合并。

In [ ]:
# ============================================================
# 5. 补充：Annotated + operator.add —— 「追加式」字段
# ============================================================
class ChatState(TypedDict):
    """演示追加式字段的状态。

    | 写法                                  | 合并行为                | 典型用途          |
    |---------------------------------------|-------------------------|-------------------|
    | title: str                            | 覆盖：新值盖旧值        | 当前状态类字段    |
    | history: Annotated[list, operator.add]| 追加：新旧列表拼接      | 对话历史、执行日志|
    | messages: Annotated[list, add_messages]| 追加 + 按 id 去重/更新 | LangGraph 内置    |

    注意 `operator.add` 作用在 list 上就是 `[] + []`（列表拼接），
    作用在 int/str 上则是数值相加 / 字符串相连——所以 reducer 选错类型会出怪事。
    """

    title: str  # 普通字段：覆盖
    history: Annotated[list[str], operator.add]  # 追加式字段：自动 concat

注意看下面两个节点：**写法与第 2 节的 `node_a` / `node_b` 完全一样**，
都只返回「自己那一条」。区别**只来自状态字段的声明方式**。

In [ ]:
def node_a(state: ChatState) -> dict:
    """节点 A：只返回「我这一条」日志，不操心拼接。"""
    # 对比上面 step_two 的手工拼接——这里只写自己新增的部分
    return {"title": "A 处理后的标题", "history": ["经过节点A"]}


def node_b(state: ChatState) -> dict:
    """节点 B：同样只返回自己那一条。"""
    return {"title": "B 处理后的标题", "history": ["经过节点B"]}


annotated_builder = StateGraph(ChatState)
annotated_builder.add_node("a", node_a)
annotated_builder.add_node("b", node_b)
annotated_builder.add_edge(START, "a")
annotated_builder.add_edge("a", "b")
annotated_builder.add_edge("b", END)
annotated_graph = annotated_builder.compile()

In [ ]:
def run_annotated_demo() -> None:
    print("=" * 72)
    print("④ 补充：Annotated + operator.add 的「追加式」字段")
    print("=" * 72)
    result = annotated_graph.invoke({"title": "初始标题", "history": ["图开始执行"]})
    print("  返回：", result)
    # 预期输出：
    #   title   → "B 处理后的标题"（覆盖式：A 写的被 B 盖掉了，只剩最后一句）
    #   history → ['图开始执行', '经过节点A', '经过节点B']（追加式：一条都没丢）
    print()
    print("  对照结论：同一个状态里，title 走覆盖、history 走追加；")
    print("  差别**只来自字段的声明方式**，节点函数本身的写法完全一样。")
    print()


run_annotated_demo()

### 预期输出

```text
  返回： {'title': 'B 处理后的标题', 'history': ['图开始执行', '经过节点A', '经过节点B']}
```

对照着看这一张表，就明白 reducer 到底做了什么：

| 字段 | 初值 | 节点 A 返回 | 节点 B 返回 | 最终值 | 为什么 |
|---|---|---|---|---|---|
| `title` | `"初始标题"` | `"A 处理后的标题"` | `"B 处理后的标题"` | `"B 处理后的标题"` | 普通字段 → **覆盖** |
| `history` | `['图开始执行']` | `['经过节点A']` | `['经过节点B']` | 三条全在 | `operator.add` → **追加** |

## 小结

- **状态**是 `TypedDict`，它的字段声明同时决定了「有没有」和「怎么合并」；
- **节点**是普通函数，**返回增量而不是全量**；
- **普通边**写死跳转，**条件边**由路由函数的返回值决定；
- **路由函数只返回名字**，跳转由框架按第三个参数的映射表完成；
- **reducer**（`Annotated[..., operator.add]`）让「追加」变成声明式，
  同一份节点代码，换个字段声明就从覆盖变追加。

下一课 `02_记忆_短期与长期.ipynb` 会看到：把 `operator.add` 换成 LangGraph 内置的
`add_messages`，再加一个 checkpointer，图就**记得住上一轮对话**了。

## 常见坑

1. **节点返回的必须是 dict（增量）**，不是完整 state。返回整个 state 不会报错，
   但 reducer 的语义会变得难以预料。
2. **`compile()` 之后图结构就冻结了**：再 `add_node` / `add_edge` 不会生效、
   **也不会报错** —— 要改结构必须改 builder 后重新 `compile()`。
3. **条件边的路由函数不执行跳转**，它只返回名字；跳转靠映射表。
4. **状态字段没声明却返回同名键，会被静默丢弃**（TypedDict 不做运行时校验）。
   排查「数据莫名不见了」时，先回来核对 schema。
5. **`graph.get_graph()` 画出的条件边依赖第三个参数**：不传映射/列表时，
   条件边在图上就是残缺的 —— 这是「打印图结构」时最常见的困惑来源。

## 官方链接

- 图 API（StateGraph / Node / Edge / Command）：<https://docs.langchain.com/oss/python/langgraph/graph-api>
- 快速上手：<https://docs.langchain.com/oss/python/langgraph/quickstart>
- 用图 API 构建工作流：<https://docs.langchain.com/oss/python/langgraph/use-graph-api>
- 函数式 API（本课的另一种写法，见 `10_控制流与函数式API_官方补充`）：<https://docs.langchain.com/oss/python/langgraph/functional-api>